# Ứng dụng thực tiễn — Phân tích Giỏ Hàng (Market Basket Analysis)

## Mục tiêu
Áp dụng thuật toán **FP-Growth** (from Scratch by Julia) nhằm để:
1. Khai thác các **Frequent Itemsets** từ dữ liệu bán lẻ thực tế (**Groceries Dataset** — 3.898 khách hàng, 167 sản phẩm)
2. Tạo sinh **Association Rules** với ngưỡng `sup ≥ minsup` và `conf(X⇒Y) ≥ minconf`
3. Trình bày **Top-10 luật theo Lift** và giải thích ý nghĩa kinh doanh

**Chiến lược gom giỏ hàng:** Mỗi *khách hàng* (`Member_number`) được xem là một giao dịch — bao gồm **toàn bộ lịch sử mua hàng** của họ. Cách này phù hợp với bài toán gợi ý sản phẩm theo hành vi tích lũy của từng người dùng.


## 1. Thiết lập môi trường

In [1]:
using Pkg
Pkg.instantiate()

include("../src/FPGrowth.jl")
using .FPGrowth
include("../src/logger.jl")
include("../src/utils.jl")
using .Utils

using CSV
using DataFrames
using Plots
using Statistics
using Plots.PlotMeasures

logger = Logger()
phase(logger, "Market Basket Analysis — FP-Growth")


┌ Warning: The project dependencies or compat requirements have changed since the manifest was last resolved.
│ It is recommended to `Pkg.resolve()` or consider `Pkg.update()` if necessary.
└ @ Pkg.API C:\Users\ADMIN\.julia\juliaup\julia-1.12.6+0.x64.w64.mingw32\share\julia\stdlib\v1.12\Pkg\src\API.jl:1322



__________________________________________________
[phase]  Market Basket Analysis — FP-Growth
__________________________________________________


## 2. Tiền xử lý Dữ liệu

| Thông tin | Chi tiết |
|---|---|
| Nguồn | Groceries Dataset (Kaggle) |
| Số hàng gốc | 38.765 hành vi mua hàng |
| Số khách hàng | 3.898 (`Member_number`) |
| Số sản phẩm | 167 loại |
| Đơn vị giao dịch | Một lượt mua hàng theo `(Member_number, Date)` |

**Chiến lược:** Gom theo `(Member_number, Date)` — mỗi lượt ghé siêu thị là một giao dịch.
Cách này phản ánh đúng hành vi mua đồng thời, phù hợp với bài toán Market Basket Analysis.


In [ ]:
# Đọc dữ liệu thô
DATA_PATH = "../data/analysis/Groceries_dataset.csv"
raw_df = CSV.read(DATA_PATH, DataFrame)

info(logger, "Raw dataset: ", nrow(raw_df), " rows | ",
     length(unique(raw_df.Member_number)), " customers | ",
     length(unique(raw_df.itemDescription)), " unique products")

first(raw_df, 5)


In [ ]:
# ── Mã hóa sản phẩm → Integer ID ────────────────────────────────────────────
all_items = sort(unique(raw_df.itemDescription))
item2id   = Dict(item => i for (i, item) in enumerate(all_items))
id2item   = Dict(i => item for (i, item) in enumerate(all_items))
n_products = length(all_items)
info(logger, "Encoded ", n_products, " products → Integer IDs")

# ── Gom theo Member_number (lịch sử mua hàng đầy đủ của mỗi khách) ──────────
grouped = groupby(raw_df, [:Member_number, :Date])
transactions = Vector{Vector{Int}}()
for g in grouped
    basket = sort(unique([item2id[row.itemDescription] for row in eachrow(g)]))
    isempty(basket) || push!(transactions, basket)
end

n_tx    = length(transactions)
avg_len = round(mean(length.(transactions)), digits=2)
max_len = maximum(length.(transactions))

info(logger, "Transactions (customers): ", n_tx)
info(logger, "Avg basket size: ", avg_len, " products | Max: ", max_len, " products")
println("\nSample basket [customer 1]: ", [id2item[i] for i in transactions[1]])


In [ ]:
# ── Thống kê phân phối kích thước giỏ hàng ──────────────────────────────────
basket_sizes = length.(transactions)
p_dist = histogram(
    basket_sizes, bins=20,
    title="Distribution of Basket Sizes (per Visit: Customer × Date)",
    xlabel="Number of Unique Products",
    ylabel="Number of Customers",
    color=:steelblue, alpha=0.8, legend=false,
    left_margin=5mm
)
vline!(p_dist, [avg_len], color=:red, linewidth=2,
       linestyle=:dash, label="Mean = $avg_len")
display(p_dist)


## 3. Khai thác Frequent Itemsets bằng FP-Growth

In [ ]:
# ── Cấu hình thực nghiệm ─────────────────────────────────────────────────────
MIN_SUP_RATIO = 0.001   # 0.1% — phù hợp với basket-per-visit (sparse baskets)
MIN_CONF      = 0.05    # Confidence tối thiểu 5%

min_sup_abs = ceil(Int, MIN_SUP_RATIO * n_tx)
info(logger, "MinSup = $(MIN_SUP_RATIO * 100)%  → ≥ ", min_sup_abs, " transactions")
info(logger, "MinConf = $(MIN_CONF * 100)%")

# ── Chạy FP-Growth Optimized (cài đặt của nhóm) ──────────────────────────────
process(logger, "Running FP-Growth Optimized...")
GC.gc()
t0  = time_ns()
freq_itemsets = FPGrowth.fpgrowth_opt(transactions, min_sup_abs)
t1  = time_ns()
exec_sec = round((t1 - t0) / 1e9, digits=3)

success(logger, "Mined ", length(freq_itemsets), " frequent itemsets in ", exec_sec, "s")

# Phân bố theo kích thước
size_dist = Dict{Int,Int}()
for k in keys(freq_itemsets); s = length(k); size_dist[s] = get(size_dist, s, 0) + 1; end
for s in sort(collect(keys(size_dist)))
    metric(logger, "  Size-$s itemsets: ", size_dist[s])
end


In [ ]:
# ── Top-15 sản phẩm phổ biến nhất ───────────────────────────────────────────
freq_df = DataFrame(
    Itemset      = [join([id2item[i] for i in k], " + ") for k in keys(freq_itemsets)],
    Size         = [length(k) for k in keys(freq_itemsets)],
    Count        = [v for v in values(freq_itemsets)],
    SupportPct   = [round(v / n_tx * 100, digits=2) for v in values(freq_itemsets)]
)
sort!(freq_df, :Count, rev=true)

top_single = first(filter(r -> r.Size == 1, freq_df), 15)

p_items = bar(
    top_single.Itemset, top_single.SupportPct,
    title  = "Top-15 Most Frequent Products",
    xlabel = "Product",
    ylabel = "Support (%)",
    color  = :steelblue, legend = false,
    xrotation = 45, bar_width = 0.7, alpha = 0.85,
    bottom_margin = 18mm, left_margin = 5mm, size = (900, 480)
)
display(p_items)


## 4. Sinh Association Rules

Từ các frequent itemsets, ta sinh luật $X \Rightarrow Y$ với 3 chỉ số đánh giá:

| Chỉ số | Công thức | Ý nghĩa |
|---|---|---|
| **Support** | $\text{sup}(X \cup Y) / N$ | Tỷ lệ giao dịch chứa cả $X$ và $Y$ |
| **Confidence** | $\text{sup}(X \cup Y) / \text{sup}(X)$ | Xác suất mua $Y$ khi đã mua $X$ |
| **Lift** | $\text{conf}(X \Rightarrow Y) / \text{sup}(Y)$ | Lift > 1: tương quan dương thực sự |

In [ ]:
function generate_association_rules(
        freq_itemsets::Dict{Vector{Int}, Int},
        n_transactions::Int,
        min_conf::Float64
)
    rules = NamedTuple{(:antecedent, :consequent, :support, :confidence, :lift),
                       Tuple{Vector{Int}, Vector{Int}, Float64, Float64, Float64}}[]
    sup_map = Dict(k => v / n_transactions for (k, v) in freq_itemsets)

    for (itemset, cnt) in freq_itemsets
        length(itemset) < 2 && continue
        itemset_sup = cnt / n_transactions
        n = length(itemset)
        for mask in 1:(2^n - 2)
            ante = Int[]; cons = Int[]
            for j in 1:n
                if ((mask >> (j-1)) & 1) == 1
                    push!(ante, itemset[j])
                else
                    push!(cons, itemset[j])
                end
            end
            isempty(cons) && continue
            ante_sup = get(sup_map, sort(ante), 0.0)
            ante_sup == 0.0 && continue
            conf = itemset_sup / ante_sup
            conf < min_conf && continue
            cons_sup = get(sup_map, sort(cons), 0.0)
            lift = cons_sup > 0.0 ? conf / cons_sup : 0.0
            push!(rules, (antecedent=sort(ante), consequent=sort(cons),
                          support=itemset_sup, confidence=conf, lift=lift))
        end
    end
    rules
end

process(logger, "Generating association rules...")
rules = generate_association_rules(freq_itemsets, n_tx, MIN_CONF)
success(logger, "Generated ", length(rules), " association rules")

# DataFrame
association_rules = DataFrame(
    Antecedent  = [join([id2item[i] for i in r.antecedent], " + ") for r in rules],
    Consequent  = [join([id2item[i] for i in r.consequent], " + ") for r in rules],
    Support_pct = [round(r.support * 100, digits=3) for r in rules],
    Confidence_pct = [round(r.confidence * 100, digits=2) for r in rules],
    Lift        = [round(r.lift, digits=4) for r in rules]
)
sort!(association_rules, :Lift, rev=true)
println("Quantity of rules: ", nrow(association_rules))


## 5. Top-10 Luật theo Lift

In [ ]:
top10 = first(association_rules, 10)

phase(logger, "Top-10 Association Rules by Lift")
println("\n", repeat("─", 102))
println(rpad("#",4), rpad("Antecedent",33), rpad("Consequent",30),
        rpad("Support%",12), rpad("Confident%",12), "Lift")
println(repeat("─", 102))
for (i, row) in enumerate(eachrow(top10))
    println(rpad(i,4),
            rpad(row.Antecedent,33),
            rpad(row.Consequent,30),
            rpad("$(row.Support_pct)%",12),
            rpad("$(row.Confidence_pct)%",12),
            row.Lift)
end
println(repeat("─", 102))


## 6. Trực quan hóa

In [ ]:
# ── Biểu đồ 2: Support vs Confidence, màu theo Lift ─────────────────────────
n_show = min(100, nrow(association_rules))
sub_df = first(association_rules, n_show)
p_scatter = scatter(
    sub_df.Support_pct, sub_df.Confidence_pct,
    marker_z    = sub_df.Lift,
    color        = :plasma,
    markersize   = 5, markerstrokewidth = 0,
    colorbar     = :left,
    colorbar_title = "Lift",
    title  = "Support vs Confidence (Top-$n_show rules, color = Lift)",
    xlabel = "Support (%)",
    ylabel = "Confidence (%)",
    legend = false,
    left_margin = 20mm, bottom_margin = 5mm
)
display(p_scatter)


In [ ]:
# ── Biểu đồ 3: Top-10 Rules — Horizontal Bar ────────────────────────────────
labels10 = ["$(r.Antecedent) → $(r.Consequent)" for r in eachrow(top10)]
p_top10 = bar(
    reverse(top10.Lift),
    yticks = (1:10, reverse(labels10)),
    orientation = :horizontal,
    title   = "Top-10 Rules by Lift",
    xlabel  = "Lift",
    color   = palette(:viridis, 10),
    legend  = false,
    bar_width = 0.6,
    left_margin = 75mm, 
    bottom_margin = 5mm,
    size = (1000, 500)
)
display(p_top10)


In [ ]:
# ── Biểu đồ 4: Association Rule Network Graph ───────────────────────────────
unique_items = String[]
for row in eachrow(top10)
    append!(unique_items, split(row.Antecedent, " + "))
    append!(unique_items, split(row.Consequent, " + "))
end
unique_items = unique(unique_items)
n_nodes = length(unique_items)

# Tọa độ vòng tròn của các node
theta = range(0, 2π, length=n_nodes+1)[1:end-1]
x_coords = cos.(theta)
y_coords = sin.(theta)
node_map = Dict(unique_items[i] => (x_coords[i], y_coords[i]) for i in 1:n_nodes)

# Vẽ các node
p_graph = scatter(
    x_coords, y_coords,
    markercolor = :lightblue,
    markersize = 22,
    markerstrokewidth = 2,
    markerstrokecolor = :steelblue,
    legend = false,
    aspect_ratio = :equal,
    title = "Association Rule Network Graph (Top-10 Rules)",
    axis = false,
    ticks = false,
    grid = false,
    xlims = (-1.5, 1.5),
    ylims = (-1.5, 1.5),
    size = (800, 800)
)

# Thêm nhãn cho các node (dịch chuyển ra ngoài vòng tròn một chút)
for i in 1:n_nodes
    ox = x_coords[i] * 1.3
    oy = y_coords[i] * 1.3
    annotate!(p_graph, ox, oy, text(unique_items[i], 9, :black, :center))
end

# Vẽ các cạnh đại diện cho luật (Antecedent -> Consequent)
for row in eachrow(top10)
    antes = split(row.Antecedent, " + ")
    conses = split(row.Consequent, " + ")
    for a in antes, c in conses
        xA, yA = node_map[a]
        xC, yC = node_map[c]
        dx = xC - xA
        dy = yC - yA
        dist = sqrt(dx^2 + dy^2)
        if dist > 0
            # Rút ngắn đường vẽ để không đè lên vòng tròn của node
            x_start = xA + 0.12 * (dx / dist)
            x_end = xC - 0.12 * (dx / dist)
            y_start = yA + 0.12 * (dy / dist)
            y_end = yC - 0.12 * (dy / dist)
            
            # Độ dày nét vẽ tỷ lệ với Lift
            lw = 1.0 + (row.Lift - minimum(top10.Lift)) / (maximum(top10.Lift) - minimum(top10.Lift)) * 3.5
            
            plot!(p_graph, [x_start, x_end], [y_start, y_end],
                  arrow = :arrow,
                  linecolor = :coral,
                  linewidth = lw,
                  alpha = 0.8,
                  label = ""
            )
        end
    end
end

display(p_graph)


In [ ]:
# ── Biểu đồ 5: Parallel Coordinates Plot ────────────────────────────────────
n_items = length(unique_items)
item_y = Dict(unique_items[i] => Float64(i) for i in 1:n_items)

p_coords = plot(
    [1, 1], [1, n_items], linecolor=:grey, linewidth=2, label="",
    xlims = (0.5, 2.5), ylims = (0.5, n_items + 0.5),
    xticks = ([1, 2], ["LHS (Antecedent)", "RHS (Consequent)"]),
    yticks = (1:n_items, unique_items),
    legend = false,
    title = "Parallel Coordinates Plot (Top-10 Rules)",
    left_margin = 15mm,
    size = (800, 600)
)
plot!(p_coords, [2, 2], [1, n_items], linecolor=:grey, linewidth=2, label="")

# Vẽ các đường nối cho mỗi luật
for row in eachrow(top10)
    antes = split(row.Antecedent, " + ")
    conses = split(row.Consequent, " + ")
    for a in antes, c in conses
        yA = item_y[a]
        yC = item_y[c]
        
        # Độ dày nét tỷ lệ với Lift
        lw = 1.0 + (row.Lift - minimum(top10.Lift)) / (maximum(top10.Lift) - minimum(top10.Lift)) * 4.0
        
        plot!(p_coords, [1, 2], [yA, yC],
              linewidth = lw,
              linecolor = :tomato,
              alpha = 0.7,
              label = ""
        )
    end
end

display(p_coords)


## 7. Giải thích Ý nghĩa Kinh doanh

### 7.1 Đọc kết quả Top-10

Tất cả 10 luật có Lift ≈ 1.51–1.61, cho thấy sự tương quan **dương thực sự** giữa các nhóm sản phẩm. Cụ thể:

| Luật tiêu biểu | Ý nghĩa kinh doanh |
|---|---|
| `other vegetables + whole milk → rolls/buns + yogurt` | Khách mua rau và sữa tươi có xu hướng mua thêm bánh và sữa chua — **combo bữa sáng hoàn chỉnh** |
| `sausage → rolls/buns + yogurt` | Xúc xích đi kèm bánh mì và sữa chua — gợi ý **set sandwich sáng** |
| `rolls/buns + sausage → yogurt` | Confidence 43% — gần 1/2 khách mua bánh+xúc xích cũng mua sữa chua |

### 7.2 Ứng dụng thực tiễn

| Ứng dụng | Mô tả |
|---|---|
| **Cross-selling** | Hiển thị gợi ý *"Khách hàng cũng mua..."* dựa trên Lift cao nhất |
| **Bundle promotion** | Thiết kế combo giảm giá cho các cặp có Conf cao (ví dụ: bánh mì + xúc xích) |
| **Store layout** | Đặt kệ rau - sữa tươi - bánh - sữa chua gần nhau để tăng khả năng mua kèm |
| **Inventory sync** | Khi tồn kho `whole milk` thấp, ưu tiên bổ sung thêm `rolls/buns` và `yogurt` |

### 7.3 Nhận xét kỹ thuật
- **MinSup = 0.1%** phù hợp cho basket-per-visit (phân tích lượt mua đồng thời): mỗi giỏ hàng chỉ có trung bình 2.59 sản phẩm nên ngưỡng thấp là cần thiết để khai thác được các mẫu ý nghĩa.
- Các luật có Lift cao tập trung quanh nhóm **thực phẩm bữa sáng** (sữa, bánh mì, sữa chua, xúc xích) — nhất quán với hành vi mua sắm thực tế.
- Điều này chứng tỏ FP-Growth Optimized hoạt động **chính xác và hiệu quả** trên dữ liệu thực tế.

In [ ]:
# ── Thống kê tổng hợp ────────────────────────────────────────────────────────
phase(logger, "Summary")
metric(logger, "Dataset            : Groceries | $(n_tx) customers | $(n_products) products")
metric(logger, "Minimum Support             : $(MIN_SUP_RATIO * 100)%  (≥ $(min_sup_abs) customers)")
metric(logger, "Minimum Confident            : $(MIN_CONF * 100)%")
metric(logger, "Frequent itemsets  : $(length(freq_itemsets))")
metric(logger, "Association rules  : $(nrow(association_rules))")
metric(logger, "Time Execution     : $(exec_sec)s")
metric(logger, "Maximum Lift           : $(maximum(association_rules.Lift))")
metric(logger, "Mean Lift          : $(round(mean(association_rules.Lift), digits=4))")
metric(logger, "Rules with Lift > 1  : $(count(r -> r > 1.0, association_rules.Lift)) / $(nrow(association_rules))")
